# True Parametric PINN for full impact period (e.g., 50 impacts)

Train one model on multiple non-parametric PINN trajectories and predict an unseen parameter case over the full horizon.

This notebook also saves weights/training logs and compares unseen prediction vs true non-parametric result.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'parametric_PINN'))

from true_parametric_pinn import (
    ParametricModelConfig,
    TrueParametricPINN,
    build_supervised_dataset,
)
from parametric_pinn_force_sweep import (
    StructuralConfig,
    TrainingConfig,
    sample_phi_cases,
    simulate_case,
)


In [ ]:
# ---------------- USER SETTINGS ----------------
# Parameter domain requested by user
phi1_min, phi1_max = 1.0, 2.0
phi2_min, phi2_max = 10.0, 20.0

# Number of training samples in 2D parameter space
# Typical choices: 10, 20, 100 (larger => better coverage, more runtime)
N_samples = 20

# Impact simulation / PINN-per-segment settings used to generate trajectories
n_iter_per_seg = 800
n_segments = 50
n_points_per_seg = 600

# Parametric NN training settings
n_iter = 4000
print_every = 400
checkpoint_dir = '../Result/parametric_model_ckpt'
checkpoint_tag = 'true_parametric_full_impact_from_sweep'

# Evaluation points
# in-range interpolation
in_range_eval = (1.6, 16.0)
# optional out-of-range extrapolation (quality not guaranteed)
out_of_range_eval = (2.2, 22.0)


In [ ]:
# 1) Sample phi-cases in the requested 2D parameter domain
phi_cases = sample_phi_cases(
    n_samples=N_samples,
    phi1_min=phi1_min,
    phi1_max=phi1_max,
    phi2_min=phi2_min,
    phi2_max=phi2_max,
    seed=1234,
)

print(f'Sampled {len(phi_cases)} training cases.')
print('First 5 samples:', phi_cases[:5])

# 2) Generate training trajectories directly (no existing dataset required)
structural_cfg = StructuralConfig()
training_cfg = TrainingConfig(
    n_iter_per_seg=n_iter_per_seg,
    n_segments=n_segments,
    n_points_per_seg=n_points_per_seg,
)

t_list, x_list = [], []
for idx, (p1, p2) in enumerate(phi_cases, start=1):
    print(f'[{idx}/{len(phi_cases)}] Simulating case phi1={p1:.4f}, phi2={p2:.4f}')
    t_total, x_total = simulate_case(p1, p2, structural_cfg, training_cfg)
    t_list.append(t_total)
    x_list.append(x_total)

# 3) Build supervised dataset for true parametric model
X_data, Y_data = build_supervised_dataset(t_list, x_list, phi_cases)
t_max = max(float(np.max(t)) for t in t_list)
n_dof = x_list[0].shape[1]

cfg = ParametricModelConfig(
    n_dof=n_dof,
    m_x=structural_cfg.m_x,
    k=structural_cfg.k,
    c=structural_cfg.c,
    seg_window=t_max,
    phi1_min=phi1_min,
    phi1_max=phi1_max,
    phi2_min=phi2_min,
    phi2_max=phi2_max,
    layers=(3, 64, 64, n_dof),
    beta_data=1.0,
    lr=1e-3,
)

print('X_data shape:', X_data.shape)
print('Y_data shape:', Y_data.shape)
print('time range:', (0.0, cfg.seg_window))


In [ ]:
# Train once on full trajectories (50-impact horizon per sampled case)
model = TrueParametricPINN(cfg)
model.train_supervised(X_data, Y_data, n_iter=n_iter, print_every=print_every)

# Save weights + training history
model.save_checkpoint(checkpoint_dir, tag=checkpoint_tag)
print('Saved checkpoint to:', checkpoint_dir)


In [ ]:
# Evaluate on unseen in-range and optional out-of-range cases
def evaluate_case(phi_case):
    p1, p2 = phi_case
    t_true, x_true = simulate_case(p1, p2, structural_cfg, training_cfg)
    x_pred, _ = model.predict(t_true.reshape(-1, 1), phi1=p1, phi2=p2)

    rmse_per_dof = np.sqrt(np.mean((x_pred - x_true) ** 2, axis=0))
    rmse_global = float(np.sqrt(np.mean((x_pred - x_true) ** 2)))
    return t_true, x_true, x_pred, rmse_global, rmse_per_dof

t_in, x_in_true, x_in_pred, rmse_in, rmse_in_dof = evaluate_case(in_range_eval)
print(f'In-range eval phi={in_range_eval}: Global RMSE={rmse_in:.4e}')
print('In-range RMSE first 5 DOFs:', rmse_in_dof[:5])

t_out, x_out_true, x_out_pred, rmse_out, rmse_out_dof = evaluate_case(out_of_range_eval)
print(f'Out-of-range eval phi={out_of_range_eval}: Global RMSE={rmse_out:.4e}')
print('Out-of-range RMSE first 5 DOFs:', rmse_out_dof[:5])


In [ ]:
# Plot comparison (last DOF)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].plot(t_in, x_in_true[:, -1], 'k-', lw=2, label='True')
axes[0].plot(t_in, x_in_pred[:, -1], 'r--', lw=2, label='Parametric')
axes[0].set_title(f'In-range {in_range_eval}')
axes[0].set_xlabel('t')
axes[0].set_ylabel('x_last')
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].plot(t_out, x_out_true[:, -1], 'k-', lw=2, label='True')
axes[1].plot(t_out, x_out_pred[:, -1], 'r--', lw=2, label='Parametric')
axes[1].set_title(f'Out-of-range {out_of_range_eval}')
axes[1].set_xlabel('t')
axes[1].grid(alpha=0.25)
axes[1].legend()

plt.suptitle('True parametric PINN prediction over full impact horizon')
plt.tight_layout()
plt.show()


In [ ]:
# Plot training loss log
plt.figure(figsize=(7, 3.5))
plt.semilogy(model.loss_log, lw=1.6)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('True parametric PINN supervised training loss')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()
